# ReAct Prompting (Reason + Act)

Intercala **Thought** (raciocínio interno), **Action** (chamada a ferramenta: `Ferramenta[args]`) e **Observation** (resultado externo) em loop até emitir `Finish[resposta]`. O grounding iterativo — cada Observation ancora o próximo Thought em dados reais — elimina alucinações que surgiriam de um único passo sem verificação. Yao et al. (2022) demonstraram que remover o Thought (Act-Only) reduz precisão em QA multi-hop em 18–34%.

**Referência:** Yao et al. (2022) *ReAct: Synergizing Reasoning and Acting in Language Models.* arXiv:2210.03629

In [ ]:
!pip install -q --upgrade langchain-ollama langchain-core python-dotenv langchain requests


In [1]:
from pathlib import Path
import os, json, re, unicodedata
from textwrap import dedent

from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from langchain_ollama import ChatOllama

# ── Configuração local Ollama ──────────────────────────────────────────────
load_dotenv(override=True)
MODEL_NAME = "gemma3:4b"
CREATIVE_MODEL_NAME = "phi4-mini"
OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://127.0.0.1:11434")


# ── Constants ──────────────────────────────────────────────────────────────

# env_candidates = [Path("code/.env"), Path(".env")]
# env_path = next((p for p in env_candidates if p.exists()), None)
# if env_path is None:
#     env_path = Path(".env")

# load_dotenv(dotenv_path=env_path, override=True)

# ── Clientes LLM ───────────────────────────────────────────────────────────
llm = ChatOllama(
    model=MODEL_NAME,
    base_url=OLLAMA_BASE_URL,
    temperature=0,
)
llm_criativo = ChatOllama(
    model=CREATIVE_MODEL_NAME,
    base_url=OLLAMA_BASE_URL,
    temperature=0.7,
)

print(f"✓ Ollama | modelo padrão: {MODEL_NAME}")
print(f"✓ Ollama | modelo criativo: {CREATIVE_MODEL_NAME}")
print(f"✓ Ollama | base_url: {OLLAMA_BASE_URL}")
print("Configuração concluída.")

# ── Funções auxiliares ────────────────────────────────────────────────────────
def normalizar(texto: str) -> str:
    """Lowercase + strip diacritics for keyword matching."""
    texto = unicodedata.normalize("NFKD", texto.lower())
    return "".join(ch for ch in texto if not unicodedata.combining(ch))

def extrair_json(texto: str) -> dict:
    """Strip markdown fences and parse the first JSON object found."""
    texto = texto.strip()
    if texto.startswith("```"):
        texto = re.sub(r"^```(?:json)?\s*|\s*```$", "", texto, flags=re.S).strip()
    inicio = texto.find("{")
    fim    = texto.rfind("}")
    if inicio != -1 and fim != -1 and fim > inicio:
        texto = texto[inicio : fim + 1]
    return json.loads(texto)

def chamar_texto(llm_client, prompt_template, alternativa: str, **kwargs) -> str:
    """Call the LLM and return a string; fall back gracefully."""
    if llm_client is None:
        return alternativa
    try:
        bruto = (prompt_template | llm_client).invoke(kwargs)
        return getattr(bruto, "content", str(bruto)).strip()
    except Exception as exc:
        print(f"⚠ Ollama: {exc}. Usando alternativa.")
        return alternativa

def pedir_json(llm_client, prompt_template, alternativa: dict, **kwargs) -> dict:
    """Call the LLM and parse JSON; fall back gracefully."""
    if llm_client is None:
        return alternativa
    try:
        bruto = (prompt_template | llm_client).invoke(kwargs)
        return extrair_json(getattr(bruto, "content", str(bruto)))
    except Exception as exc:
        print(f"⚠ Ollama: {exc}. Usando alternativa.")
        return alternativa



✓ Ollama | modelo padrão: gemma3:4b
✓ Ollama | modelo criativo: phi4-mini
✓ Ollama | base_url: http://172.18.224.1:11434
Configuração concluída.


In [2]:
# ── Cenário: agente de cozinha ReAct ──────────────────────────────────────
# A receita de bolo de cenoura é usada como analogia:
#   → a ORDEM das ações importa
#   → os ingredientes precisam ser checados ANTES de agir
#   → a cobertura só entra DEPOIS do bolo assado

PASSOS_MASSA = [
    "1. Checar ingredientes da massa",
    "2. Bater cenoura, ovos, óleo e açúcar no liquidificador",
    "3. Misturar farinha, sal e fermento na tigela",
    "4. Incorporar as duas misturas",
    "5. Levar para a forma untada e enfarinhada",
    "6. Assar em forno preaquecido a 180°C por 40 minutos",
]
PASSOS_COBERTURA = [
    "1. Levar açúcar, chocolate em pó, manteiga e leite à panela",
    "2. Mexer até engrossar",
    "3. Cobrir o bolo depois de assado",
]
FLUXO_RECEITA = (
    "Massa:\n" + "\n".join(PASSOS_MASSA) +
    "\n\nCobertura:\n" + "\n".join(PASSOS_COBERTURA)
)
INGREDIENTES_MASSA     = ["cenoura", "ovos", "açúcar", "óleo", "farinha", "sal", "fermento"]
INGREDIENTES_COBERTURA = ["açúcar", "chocolate em pó", "manteiga", "leite"]

print("Fluxo da receita carregado.")
print(FLUXO_RECEITA)


Fluxo da receita carregado.
Massa:
1. Checar ingredientes da massa
2. Bater cenoura, ovos, óleo e açúcar no liquidificador
3. Misturar farinha, sal e fermento na tigela
4. Incorporar as duas misturas
5. Levar para a forma untada e enfarinhada
6. Assar em forno preaquecido a 180°C por 40 minutos

Cobertura:
1. Levar açúcar, chocolate em pó, manteiga e leite à panela
2. Mexer até engrossar
3. Cobrir o bolo depois de assado


## 01. Thought — raciocínio interno

O agente planeja a sequência correta **antes** de qualquer ação. Thought nunca chama ferramentas.

In [3]:
# ── 01. Thought — raciocínio interno ─────────────────────────────────────
# O agente planeja a sequência correta ANTES de qualquer ação.
# Thought nunca chama ferramentas — é puro raciocínio.

CENARIO_MASSA = (
    "Vou fazer um bolo de cenoura para 6 pessoas. "
    "Tenho os ingredientes da massa, mas devo respeitar a ordem: "
    "primeiro checar, depois bater os líquidos, misturar os secos, "
    "assar e só no fim cobrir."
)

prompt_thought = PromptTemplate(
    input_variables=["cenario", "fluxo_receita"],
    template=(
        "Você é um agente ReAct planejando uma receita de bolo de cenoura.\n"
        "Produza APENAS um bloco Thought (3 frases curtas).\n"
        "Siga a ordem: massa → forno → cobertura. Não finalize a receita.\n\n"
        "Fluxo obrigatório:\n{fluxo_receita}\n\n"
        "Cenário: {cenario}\n"
        "Thought:"
    ),
)

thought = chamar_texto(
    llm,
    prompt_thought,
    alternativa=(
        "Primeiro preciso confirmar que tenho todos os ingredientes da massa. "
        "Depois, bato cenoura, ovos, óleo e açúcar no liquidificador. "
        "Só então misturo farinha, sal e fermento — e a cobertura fica para depois do forno."
    ),
    cenario=CENARIO_MASSA,
    fluxo_receita=FLUXO_RECEITA,
)

print("Cenário:")
print(CENARIO_MASSA)
print("\nThought:")
print(thought)


Cenário:
Vou fazer um bolo de cenoura para 6 pessoas. Tenho os ingredientes da massa, mas devo respeitar a ordem: primeiro checar, depois bater os líquidos, misturar os secos, assar e só no fim cobrir.

Thought:
Primeiro, preciso verificar se tenho todos os ingredientes da massa para um bolo de cenoura para 6 pessoas. Em seguida, vou bater a cenoura, os ovos, o óleo e o açúcar no liquidificador para garantir uma massa homogênea. Depois, misturarei os ingredientes secos separadamente para evitar que o glúten se desenvolva demais.


## 02. Action — chamada de ferramenta

O agente escolhe a próxima ferramenta e seus argumentos no formato `Ferramenta[args]`.

In [4]:
# ── 02. Action — chamada de ferramenta ────────────────────────────────────
# O agente escolhe a próxima ferramenta e seus argumentos.
# Formato padronizado: Ferramenta[argumentos]

CENARIO_ACAO = (
    "A receita de bolo de cenoura está definida. "
    "Preciso confirmar a ordem de execução antes de agir. "
    "Quero começar pela massa, sem pular o forno nem a forma."
)

FERRAMENTAS_DISPONIVEIS = (
    "ChecarDespensa, PreaquecerForno, SepararForma, "
    "MisturarMassa, Assar, BuscarSubstituto, PrepararCobertura"
)

prompt_action = PromptTemplate(
    input_variables=["cenario", "ingredientes_massa", "ingredientes_cobertura", "ferramentas"],
    template=(
        "Você é um agente ReAct de cozinha.\n"
        "Produza APENAS um bloco Action no formato Ferramenta[argumentos].\n"
        "Escolha a próxima ação mais útil seguindo a ordem da receita.\n\n"
        "Ferramentas disponíveis: {ferramentas}\n"
        "Ingredientes da massa: {ingredientes_massa}\n"
        "Ingredientes da cobertura: {ingredientes_cobertura}\n\n"
        "Cenário: {cenario}\n"
        "Action:"
    ),
)

action = chamar_texto(
    llm,
    prompt_action,
    alternativa="ChecarDespensa[cenoura, ovos, açúcar, óleo, farinha, sal, fermento]",
    cenario=CENARIO_ACAO,
    ingredientes_massa=str(INGREDIENTES_MASSA),
    ingredientes_cobertura=str(INGREDIENTES_COBERTURA),
    ferramentas=FERRAMENTAS_DISPONIVEIS,
)

print("Cenário:")
print(CENARIO_ACAO)
print("\nAction:")
print(action)


Cenário:
A receita de bolo de cenoura está definida. Preciso confirmar a ordem de execução antes de agir. Quero começar pela massa, sem pular o forno nem a forma.

Action:
ChecarDespensa[massa]


## 03. Observation — resultado da ferramenta

O agente interpreta o retorno externo e decide o próximo passo.

In [5]:
# ── 03. Observation — resultado da ferramenta ─────────────────────────────
# O agente interpreta o retorno da ferramenta e decide o próximo passo.

RESULTADO_FERRAMENTA = """
ChecarDespensa:
  cenoura:        ok
  ovos:           ok
  açúcar:         ok
  óleo:           ok
  farinha:        ok
  sal:            ok
  fermento:       ok
  chocolate em pó: FALTANDO
  manteiga:       ok
  leite:          ok

PreaquecerForno:
  temperatura alvo: 180°C — ainda não atingida

Forma:
  untada e enfarinhada: não preparada
""".strip()

prompt_observation = PromptTemplate(
    input_variables=["resultado_ferramenta", "fluxo_receita"],
    template=(
        "Você é um agente ReAct.\n"
        "Converta o resultado da ferramenta em uma Observation clara.\n"
        "Diga: (1) o que está pronto, (2) o que falta, (3) próxima ação recomendada.\n\n"
        "Fluxo da receita:\n{fluxo_receita}\n\n"
        "Resultado da ferramenta:\n{resultado_ferramenta}\n"
        "Observation:"
    ),
)

observation = chamar_texto(
    llm,
    prompt_observation,
    alternativa=(
        "Pronto: ingredientes da massa verificados (todos ok).\n"
        "Falta: chocolate em pó para a cobertura; forno ainda frio; forma não preparada.\n"
        "Próxima ação: BuscarSubstituto[chocolate em pó] e PreaquecerForno[180]."
    ),
    resultado_ferramenta=RESULTADO_FERRAMENTA,
    fluxo_receita=FLUXO_RECEITA,
)

print("Resultado da ferramenta:")
print(RESULTADO_FERRAMENTA)
print("\nObservation:")
print(observation)


Resultado da ferramenta:
ChecarDespensa:
  cenoura:        ok
  ovos:           ok
  açúcar:         ok
  óleo:           ok
  farinha:        ok
  sal:            ok
  fermento:       ok
  chocolate em pó: FALTANDO
  manteiga:       ok
  leite:          ok

PreaquecerForno:
  temperatura alvo: 180°C — ainda não atingida

Forma:
  untada e enfarinhada: não preparada

Observation:
(1) Ingredientes da massa: cenoura, ovos, óleo, açúcar, farinha, sal e fermento estão disponíveis.
(2) Falta: chocolate em pó e a forma precisa ser untada e enfarinhada. O forno também não está preaquecido.
(3) Próxima ação recomendada: Preparar a forma untando e enfarinhando-a. Em seguida, preaquecer o forno a 180°C. Finalmente, adicionar chocolate em pó à lista de ingredientes.


## 04. Loop ReAct completo

Thought → Action → Observation × N, terminando em `Finish`. Toda a cadeia anterior aparece no prompt de cada rodada.

In [6]:
# ── 04. Loop ReAct completo ───────────────────────────────────────────────
# Thought → Action → Observation × N, terminando em Finish.
# O LLM gerencia o estado interno ao ver toda a cadeia anterior no prompt.

CENARIO_COMPLETO = (
    "Quero fazer um bolo de cenoura completo para 6 pessoas. "
    "A ordem importa: massa primeiro, forno depois, cobertura só no final. "
    "Atenção: falta chocolate em pó para a cobertura — "
    "preciso decidir se ajusto a receita ou busco substituto."
)

prompt_loop_react = PromptTemplate(
    input_variables=["cenario", "fluxo_receita"],
    template=(
        "Escreva um loop ReAct de 4 rodadas para o bolo de cenoura.\n"
        "Use exatamente os rótulos: "
        "Thought 1, Action 1, Observation 1, "
        "Thought 2, Action 2, Observation 2, "
        "Thought 3, Action 3, Observation 3, "
        "Thought 4, Action 4, Finish.\n"
        "Mantenha cada linha curta e técnica.\n"
        "Regra: cobertura só depois do bolo assado.\n"
        "Se chocolate em pó faltar, use BuscarSubstituto[ingrediente] antes de PrepararCobertura.\n\n"
        "Fluxo da receita:\n{fluxo_receita}\n\n"
        "Cenário: {cenario}\n"
        "ReAct:"
    ),
)

loop_react = chamar_texto(
    llm,
    prompt_loop_react,
    alternativa=(
        "Thought 1: Verificar ingredientes da massa.\n"
        "Action 1: ChecarDespensa[cenoura, ovos, açúcar, óleo, farinha, sal, fermento]\n"
        "Observation 1: Todos os ingredientes da massa estão disponíveis.\n\n"
        "Thought 2: Pré-aquecer o forno e preparar a forma.\n"
        "Action 2: PreaquecerForno[180]\n"
        "Observation 2: Forno atingindo 180°C.\n\n"
        "Thought 3: Misturar e assar a massa.\n"
        "Action 3: MisturarMassa[] → Assar[40min]\n"
        "Observation 3: Bolo assado com sucesso.\n\n"
        "Thought 4: Chocolate em pó falta — buscar substituto antes da cobertura.\n"
        "Action 4: BuscarSubstituto[chocolate em pó] → PrepararCobertura[]\n"
        "Finish: Bolo de cenoura completo com cobertura substituída."
    ),
    cenario=CENARIO_COMPLETO,
    fluxo_receita=FLUXO_RECEITA,
)

print("Cenário:")
print(CENARIO_COMPLETO)
print("\nLoop ReAct:")
print(loop_react)


Cenário:
Quero fazer um bolo de cenoura completo para 6 pessoas. A ordem importa: massa primeiro, forno depois, cobertura só no final. Atenção: falta chocolate em pó para a cobertura — preciso decidir se ajusto a receita ou busco substituto.

Loop ReAct:
Thought 1: Preciso seguir a receita do bolo de cenoura passo a passo.
Action 1: Verificar os ingredientes da massa.
Observation 1: Ingredientes da massa disponíveis: cenoura, ovos, óleo, açúcar, farinha, sal, fermento.
Thought 2: A massa parece completa, mas falta chocolate em pó para a cobertura.
Action 2: Verificar se há chocolate em pó disponível.
Observation 2: Chocolate em pó não está disponível.
Thought 3: Preciso decidir se ajusto a receita ou busco um substituto para o chocolate em pó. Ajustar a receita pode alterar o sabor.
Action 3: BuscarSubstituto[chocolate em pó].
Observation 3: Achado: Cacau em pó.
Thought 4: Cacau em pó é um bom substituto para chocolate em pó.
Action 4: PrepararMassa usando cacau em pó no lugar do choco